In [ ]:
import pandas as pd

# ==========================================
# 1. CARGA DE DATOS
# ==========================================
print("Cargando bases de datos...")
hts = pd.read_parquet("../data/parquet_v3/hts.parquet")
aux = pd.read_parquet("../data/parquet_v3/auxiliar.parquet")

# IMPORTANTE: Leemos 'code' como string desde Excel para no perder ceros a la izquierda (ej. 0101.21)
metals = pd.read_excel("../data/manual/metals_comparativa.xlsx", dtype={'code': str})

# ==========================================
# 2. LIMPIEZA DE LA BASE MANUAL
# ==========================================
# Quitamos los puntos de los códigos (ej. '4023.4' -> '40234') 
metals['code_clean'] = metals['code'].str.replace('.', '', regex=False)
metals['code_len'] = metals['code_clean'].str.len()

# Separamos en las listas 'old' y 'new'
old_list = metals[metals['category'] == 'old']
new_list = metals[metals['category'] == 'new']

# ==========================================
# 3. FUNCIÓN DE MAPEO (EXPANSIÓN POR PREFIJO)
# ==========================================
def map_annexes(target_codes, manual_df):
    """
    Expande los códigos de la lista manual hacia la base destino.
    Retorna una Serie de Pandas con el anexo correspondiente o 'out' si no aplica.
    """
    # Ordenar por longitud descendente para que las fracciones más específicas 
    # (más largas) tengan prioridad al momento de asignar.
    manual_df = manual_df.sort_values(by='code_len', ascending=False)
    
    # Inicializamos todos los códigos como "out"
    annexes = pd.Series("out", index=target_codes.index)
    
    for _, row in manual_df.iterrows():
        prefix = row['code_clean']
        annex = row['annex']
        
        # Buscamos los que empiecen con el prefijo y que aún tengan "out" 
        # (para respetar la prioridad de los códigos más específicos)
        mask = target_codes.str.startswith(prefix) & (annexes == "out")
        annexes[mask] = annex
        
    return annexes

# ==========================================
# 4. PROCESAR BASE HTS (Para <= 8 dígitos)
# ==========================================
print("Procesando base HTS (8 dígitos)...")
hts['code'] = hts['code'].astype(str)

# Filtramos las listas manuales para dejar solo fracciones de 8 dígitos o menos
old_list_hts = old_list[old_list['code_len'] <= 8]
new_list_hts = new_list[new_list['code_len'] <= 8]

# Mapeamos
hts['annex_previo'] = map_annexes(hts['code'], old_list_hts)
hts['annex_actual'] = map_annexes(hts['code'], new_list_hts)

# Nos quedamos SÓLO con los códigos que pertenezcan a algún anexo (viejo o nuevo)
hts_filtered = hts[(hts['annex_previo'] != 'out') | (hts['annex_actual'] != 'out')].copy()

# ==========================================
# 5. PROCESAR BASE AUXILIAR (Para > 8 dígitos)
# ==========================================
print("Procesando base Auxiliar (10 dígitos)...")
aux['code'] = aux['code'].astype(str)

# Filtramos para usar solo los códigos manuales de más de 8 dígitos
old_list_aux = old_list[old_list['code_len'] > 8]
new_list_aux = new_list[new_list['code_len'] > 8]

# Mapeamos
aux['annex_previo'] = map_annexes(aux['code'], old_list_aux)
aux['annex_actual'] = map_annexes(aux['code'], new_list_aux)

# Filtramos para quedarnos con los implicados
aux_filtered = aux[(aux['annex_previo'] != 'out') | (aux['annex_actual'] != 'out')].copy()

# ==========================================
# 6. UNIR Y CALCULAR EL ESTATUS FINAL
# ==========================================
print("Generando reporte final...")
# Seleccionamos las columnas de interés y las unimos
final_cols = ['code', 'annex_previo', 'annex_actual']
final_df = pd.concat([hts_filtered[final_cols], aux_filtered[final_cols]], ignore_index=True)

# Lógica para determinar qué le pasó a la fracción
def get_status(row):
    prev = row['annex_previo']
    curr = row['annex_actual']
    
    if prev == 'out' and curr != 'out':
        return 'Añadido'
    elif prev != 'out' and curr == 'out':
        return 'Quitado'
    elif prev == curr:
        return 'Mantenido'
    else:
        return 'Movido'

# Aplicamos la lógica para crear la última columna
final_df['status'] = final_df.apply(get_status, axis=1)

# Ordenamos para tener todo agrupadito por código
final_df = final_df.sort_values(by='code').reset_index(drop=True)

print("¡Listo! Primeras filas del resultado:")
print(final_df.head(15))

# Opcional: exportar a csv o excel

Cargando bases de datos...
Procesando base HTS (8 dígitos)...
Procesando base Auxiliar (10 dígitos)...
Generando reporte final...
¡Listo! Primeras filas del resultado:
          code annex_previo annex_actual     status
0     04029968     Annex II     Annex II  Mantenido
1     04029970     Annex II     Annex II  Mantenido
2     04029990     Annex II     Annex II  Mantenido
3   2106909998     Annex II     Annex II  Mantenido
4     22030000     Annex II     Annex II  Mantenido
5   2710193050     Annex II     Annex II  Mantenido
6   2711120020     Annex II     Annex II  Mantenido
7   2804290010     Annex II     Annex II  Mantenido
8     29034310     Annex II     Annex II  Mantenido
9     29034410     Annex II     Annex II  Mantenido
10    29034510     Annex II     Annex II  Mantenido
11    29034900     Annex II     Annex II  Mantenido
12    29035110     Annex II     Annex II  Mantenido
13    29035990     Annex II     Annex II  Mantenido
14    29037101          out     Annex II    Añadido


In [2]:
final_df.to_csv("analisis_fracciones_metals.csv", index=False)

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# ==========================================
# PREPARACIÓN DE DATOS (Punto 6: Diferenciar nivel)
# ==========================================
df_analisis = final_df.copy()
# Identificamos si es Fracción (8 o menos dígitos) o Producto (10 o más dígitos)
df_analisis['nivel'] = df_analisis['code'].apply(
    lambda x: 'Fracción (≤ 8 dígitos)' if len(str(x)) <= 8 else 'Producto (> 8 dígitos)'
)

# ==========================================
# FUNCIÓN DE ANÁLISIS 
# ==========================================
def reporte_cambios(df, titulo=""):
    display(Markdown(f"<br><h2>{titulo}</h2><hr>"))
    
    # 1, 2 y 4. Resumen de totales por estatus
    display(Markdown("#### Resumen General de Estatus"))
    resumen_general = df['status'].value_counts().reset_index()
    resumen_general.columns = ['Estatus', 'Total Códigos']
    display(resumen_general)
    
    # Desglose por Anexo para Añadidos, Mantenidos y Removidos
    display(Markdown("#### Desglose por Anexo (Añadidos, Mantenidos, Quitados)"))
    añadidos = df[df['status'] == 'Añadido']['annex_actual'].value_counts().to_frame('Añadidos')
    mantenidos = df[df['status'] == 'Mantenido']['annex_actual'].value_counts().to_frame('Mantenidos')
    quitados = df[df['status'] == 'Quitado']['annex_previo'].value_counts().to_frame('Quitados')
    
    # Unimos estas tres vistas en una sola tablita elegante
    resumen_anexos = pd.concat([añadidos, mantenidos, quitados], axis=1).fillna(0).astype(int)
    resumen_anexos.index.name = 'Anexo'
    display(resumen_anexos)

    # 3. Movidos (Matriz de Transición: De dónde a dónde)
    display(Markdown("#### Códigos Movidos (Matriz de Transición)"))
    df_movidos = df[df['status'] == 'Movido']
    if not df_movidos.empty:
        matriz_movidos = pd.crosstab(
            index=df_movidos['annex_previo'], 
            columns=df_movidos['annex_actual'], 
            margins=True, margins_name="Total Movidos"
        )
        matriz_movidos.index.name = 'De (Anexo Previo)'
        matriz_movidos.columns.name = 'A (Anexo Actual)'
        display(matriz_movidos)
    else:
        print("-> No hubo códigos movidos en este corte.")

    # 5. Análisis desde la Perspectiva por Anexo
    display(Markdown("#### Análisis de Composición por Anexo"))
    # Obtenemos todos los anexos únicos (ignorando los 'out')
    anexos = sorted(list(set(df['annex_previo'].unique()).union(set(df['annex_actual'].unique())) - {'out'}))
    
    for anexo in anexos:
        # Cálculos del anexo
        n_añadidos = len(df[(df['annex_actual'] == anexo) & (df['status'] == 'Añadido')])
        n_mantenidos = len(df[(df['annex_actual'] == anexo) & (df['status'] == 'Mantenido')])
        n_quitados = len(df[(df['annex_previo'] == anexo) & (df['status'] == 'Quitado')])
        
        entradas = df[(df['annex_actual'] == anexo) & (df['status'] == 'Movido')]
        salidas = df[(df['annex_previo'] == anexo) & (df['status'] == 'Movido')]
        
        total_actual = n_añadidos + n_mantenidos + len(entradas)
        
        # Imprimimos el reporte en texto estructurado
        print(f"📦 ANEXO {anexo}")
        print(f"  ■ Composición Actual: {total_actual} códigos")
        print(f"    ├─ {n_mantenidos} mantenidos")
        print(f"    ├─ {n_añadidos} añadidos nuevos")
        print(f"    └─ {len(entradas)} movidos hacia aquí")
        if not entradas.empty:
            for origen, conteo in entradas['annex_previo'].value_counts().items():
                print(f"       ↳ Vinieron desde: {origen} ({conteo} códigos)")
                
        print(f"  ■ Salidas del Anexo:")
        print(f"    ├─ {n_quitados} quitados (ya no están en ningún anexo)")
        print(f"    └─ {len(salidas)} movidos hacia otros anexos")
        if not salidas.empty:
            for destino, conteo in salidas['annex_actual'].value_counts().items():
                print(f"       ↳ Se fueron a: {destino} ({conteo} códigos)")
        print("-" * 50)

# ==========================================
# EJECUCIÓN DEL REPORTE
# ==========================================
# Análisis GLOBAL (Todo junto)
reporte_cambios(df_analisis, "ANÁLISIS GLOBAL (Fracciones y Productos combinados)")

# Análisis separado por Nivel (Fracción vs Producto)
niveles = df_analisis['nivel'].unique()
for nivel in sorted(niveles):
    # Filtramos la base por nivel y la pasamos a la función
    df_filtrado = df_analisis[df_analisis['nivel'] == nivel].copy()
    reporte_cambios(df_filtrado, f"ANÁLISIS ESPECÍFICO: {nivel}")

<br><h2>ANÁLISIS GLOBAL (Fracciones y Productos combinados)</h2><hr>

#### Resumen General de Estatus

,Estatus,Total Códigos
0,Mantenido,1053
1,Movido,66
2,Añadido,45
3,Quitado,20


#### Desglose por Anexo (Añadidos, Mantenidos, Quitados)

,Añadidos,Mantenidos,Quitados
Anexo,,,
Annex II,34,135,0
Annex I-B,9,303,20
Annex III,2,47,0
Annex I-A,0,568,0


...

,Añadidos,Mantenidos,Quitados
Anexo,,,
Annex II,11,34,0
Annex III,2,17,0
Annex I-B,2,74,19
Annex I-A,0,21,0


#### Códigos Movidos (Matriz de Transición)

A (Anexo Actual),Annex I-B,Annex III,Total Movidos
De (Anexo Previo),,,
Annex I-B,0,4,4
Annex II,1,0,1
Total Movidos,1,4,5


#### Análisis de Composición por Anexo

📦 ANEXO Annex I-A
  ■ Composición Actual: 21 códigos
    ├─ 21 mantenidos
    ├─ 0 añadidos nuevos
    └─ 0 movidos hacia aquí
  ■ Salidas del Anexo:
    ├─ 0 quitados (ya no están en ningún anexo)
    └─ 0 movidos hacia otros anexos
--------------------------------------------------
📦 ANEXO Annex I-B
  ■ Composición Actual: 77 códigos
    ├─ 74 mantenidos
    ├─ 2 añadidos nuevos
    └─ 1 movidos hacia aquí
       ↳ Vinieron desde: Annex II (1 códigos)
  ■ Salidas del Anexo:
    ├─ 19 quitados (ya no están en ningún anexo)
    └─ 4 movidos hacia otros anexos
       ↳ Se fueron a: Annex III (4 códigos)
--------------------------------------------------
📦 ANEXO Annex II
  ■ Composición Actual: 45 códigos
    ├─ 34 mantenidos
    ├─ 11 añadidos nuevos
    └─ 0 movidos hacia aquí
  ■ Salidas del Anexo:
    ├─ 0 quitados (ya no están en ningún anexo)
    └─ 1 movidos hacia otros anexos
       ↳ Se fueron a: Annex I-B (1 códigos)
--------------------------------------------------
📦 ANEXO An

In [5]:
import pandas as pd
from IPython.display import display, Markdown

# ==========================================
# PREPARACIÓN DE DATOS
# ==========================================
df_analisis = final_df.copy()

# Simplificamos los nombres para que las tablas no queden gigantes
df_analisis['nivel'] = df_analisis['code'].apply(
    lambda x: 'Fracción (≤8)' if len(str(x)) <= 8 else 'Producto (>8)'
)

display(Markdown("<br><h2>ANÁLISIS GLOBAL DETALLADO (POR FRACCIÓN Y PRODUCTO)</h2><hr>"))

# ==========================================
# 1 Y 2. RESUMEN GENERAL DE ESTATUS
# ==========================================
display(Markdown("#### 1. Resumen General de Estatus"))
# Crosstab cruza el estatus con el nivel y nos da los totales automáticos
resumen_general = pd.crosstab(
    index=df_analisis['status'], 
    columns=df_analisis['nivel'], 
    margins=True, 
    margins_name='TOTAL GENERAL'
)
# Reordenamos columnas para asegurar que el TOTAL quede al final
cols_ordenadas = [c for c in resumen_general.columns if c != 'TOTAL GENERAL'] + ['TOTAL GENERAL']
display(resumen_general[cols_ordenadas])

# ==========================================
# 4. DESGLOSE POR ANEXO Y NIVEL (Añadidos, Mantenidos, Quitados)
# ==========================================
display(Markdown("#### 2. Desglose por Anexo y Nivel (Añadidos, Mantenidos, Quitados)"))
# Excluimos los movidos temporalmente para esta tabla plana
df_amq = df_analisis[df_analisis['status'].isin(['Añadido', 'Mantenido', 'Quitado'])].copy()

# Determinamos el anexo de referencia (actual para añadidos/mantenidos, previo para quitados)
df_amq['Anexo_Ref'] = df_amq.apply(
    lambda r: r['annex_previo'] if r['status'] == 'Quitado' else r['annex_actual'], 
    axis=1
)

resumen_anexos = pd.crosstab(
    index=df_amq['Anexo_Ref'], 
    columns=[df_amq['status'], df_amq['nivel']], # Columnas anidadas (MultiIndex)
    margins=True, 
    margins_name='TOTAL GENERAL'
).fillna(0).astype(int)

resumen_anexos.index.name = 'Anexo'
display(resumen_anexos)

# ==========================================
# 3. MATRIZ DE MOVIDOS POR NIVEL
# ==========================================
display(Markdown("#### 3. Códigos Movidos (De dónde a dónde, por Nivel)"))
df_movidos = df_analisis[df_analisis['status'] == 'Movido']

if not df_movidos.empty:
    matriz_movidos = pd.crosstab(
        index=[df_movidos['annex_previo'], df_movidos['annex_actual']], 
        columns=df_movidos['nivel'], 
        margins=True, 
        margins_name="TOTAL"
    )
    matriz_movidos.index.names = ['De (Anexo Previo)', 'A (Anexo Actual)']
    display(matriz_movidos)
else:
    print("-> No hubo códigos movidos en este corte.\n")

# ==========================================
# 5 Y 6. ÁRBOL DE COMPOSICIÓN POR ANEXO
# ==========================================
display(Markdown("#### 4. Análisis de Composición por Anexo (Árbol de Detalle)"))
anexos = sorted(list(set(df_analisis['annex_previo'].unique()).union(set(df_analisis['annex_actual'].unique())) - {'out'}))

# Funciones auxiliares para formatear el texto del árbol
def get_counts(df_subset):
    f = len(df_subset[df_subset['nivel'] == 'Fracción (≤8)'])
    p = len(df_subset[df_subset['nivel'] == 'Producto (>8)'])
    return f + p, f, p

def format_counts(t, f, p):
    return f"{t} totales ({f} fracciones, {p} productos)"

for anexo in anexos:
    # Filtros base
    añadidos = df_analisis[(df_analisis['annex_actual'] == anexo) & (df_analisis['status'] == 'Añadido')]
    mantenidos = df_analisis[(df_analisis['annex_actual'] == anexo) & (df_analisis['status'] == 'Mantenido')]
    quitados = df_analisis[(df_analisis['annex_previo'] == anexo) & (df_analisis['status'] == 'Quitado')]
    entradas = df_analisis[(df_analisis['annex_actual'] == anexo) & (df_analisis['status'] == 'Movido')]
    salidas = df_analisis[(df_analisis['annex_previo'] == anexo) & (df_analisis['status'] == 'Movido')]
    
    # Cálculos
    t_añ, f_añ, p_añ = get_counts(añadidos)
    t_man, f_man, p_man = get_counts(mantenidos)
    t_qui, f_qui, p_qui = get_counts(quitados)
    t_ent, f_ent, p_ent = get_counts(entradas)
    t_sal, f_sal, p_sal = get_counts(salidas)
    
    total_actual = t_añ + t_man + t_ent
    
    # Impresión del árbol
    print(f"📦 ANEXO {anexo}")
    print(f"  ■ Composición Actual: {total_actual} códigos")
    print(f"    ├─ Mantenidos: {format_counts(t_man, f_man, p_man)}")
    print(f"    ├─ Añadidos:   {format_counts(t_añ, f_añ, p_añ)}")
    print(f"    └─ Entrantes:  {format_counts(t_ent, f_ent, p_ent)}")
    
    if not entradas.empty:
        for origen in entradas['annex_previo'].unique():
            sub_ent = entradas[entradas['annex_previo'] == origen]
            st, sf, sp = get_counts(sub_ent)
            print(f"       ↳ Desde {origen}: {format_counts(st, sf, sp)}")
            
    print(f"  ■ Salidas (Impacto respecto a lista previa):")
    print(f"    ├─ Quitados:   {format_counts(t_qui, f_qui, p_qui)}")
    print(f"    └─ Salientes:  {format_counts(t_sal, f_sal, p_sal)}")
    
    if not salidas.empty:
        for destino in salidas['annex_actual'].unique():
            sub_sal = salidas[salidas['annex_actual'] == destino]
            st, sf, sp = get_counts(sub_sal)
            print(f"       ↳ Hacia {destino}: {format_counts(st, sf, sp)}")
    print("-" * 75)

<br><h2>ANÁLISIS GLOBAL DETALLADO (POR FRACCIÓN Y PRODUCTO)</h2><hr>

#### 1. Resumen General de Estatus

nivel,Fracción (≤8),Producto (>8),TOTAL GENERAL
status,,,
Añadido,30,15,45
Mantenido,907,146,1053
Movido,61,5,66
Quitado,1,19,20
TOTAL GENERAL,999,185,1184


#### 2. Desglose por Anexo y Nivel (Añadidos, Mantenidos, Quitados)

status              Añadido                   Mantenido                \
nivel         Fracción (≤8) Producto (>8) Fracción (≤8) Producto (>8)   
Anexo                                                                   
Annex I-A                 0             0           547            21   
Annex I-B                 7             2           229            74   
Annex II                 23            11           101            34   
Annex III                 0             2            30            17   
TOTAL GENERAL            30            15           907           146   

status              Quitado               TOTAL GENERAL  
nivel         Fracción (≤8) Producto (>8)                
Anexo                                                    
Annex I-A                 0             0           568  
Annex I-B                 1            19           332  
Annex II                  0             0           169  
Annex III                 0             0            49  
TOTAL GENERAL             1            19          1118

#### 3. Códigos Movidos (De dónde a dónde, por Nivel)

nivel                               Fracción (≤8)  Producto (>8)  TOTAL
De (Anexo Previo) A (Anexo Actual)                                     
Annex I-B         Annex I-C                    28              0     28
                  Annex III                    33              4     37
Annex II          Annex I-B                     0              1      1
TOTAL                                          61              5     66

#### 4. Análisis de Composición por Anexo (Árbol de Detalle)

📦 ANEXO Annex I-A
  ■ Composición Actual: 568 códigos
    ├─ Mantenidos: 568 totales (547 fracciones, 21 productos)
    ├─ Añadidos:   0 totales (0 fracciones, 0 productos)
    └─ Entrantes:  0 totales (0 fracciones, 0 productos)
  ■ Salidas (Impacto respecto a lista previa):
    ├─ Quitados:   0 totales (0 fracciones, 0 productos)
    └─ Salientes:  0 totales (0 fracciones, 0 productos)
---------------------------------------------------------------------------
📦 ANEXO Annex I-B
  ■ Composición Actual: 313 códigos
    ├─ Mantenidos: 303 totales (229 fracciones, 74 productos)
    ├─ Añadidos:   9 totales (7 fracciones, 2 productos)
    └─ Entrantes:  1 totales (0 fracciones, 1 productos)
       ↳ Desde Annex II: 1 totales (0 fracciones, 1 productos)
  ■ Salidas (Impacto respecto a lista previa):
    ├─ Quitados:   20 totales (1 fracciones, 19 productos)
    └─ Salientes:  65 totales (61 fracciones, 4 productos)
       ↳ Hacia Annex III: 37 totales (33 fracciones, 4 productos)
       ↳ 

In [6]:
import pandas as pd
import numpy as np

df01_hts = pd.read_parquet("../data/parquet_v3/hts.parquet")
df02_aux = pd.read_parquet("../data/parquet_v3/auxiliar.parquet")
df03_metals = pd.read_excel("../data/manual/metals_comparativa.xlsx", dtype={'code': str})

df03_metals['code_clean'] = df03_metals['code'].str.replace('.', '', regex=False)
df03_metals['code_len'] = df03_metals['code_clean'].str.len()

df04_old = df03_metals[df03_metals['category'] == 'old']
df05_new = df03_metals[df03_metals['category'] == 'new']

def map_annexes(df_target, df_manual):
    df_manual = df_manual.sort_values(by='code_len', ascending=False)
    series01_annexes = pd.Series("out", index=df_target.index)
    for _, row01 in df_manual.iterrows():
        str01_prefix = row01['code_clean']
        str02_annex = row01['annex']
        mask01 = df_target.str.startswith(str01_prefix) & (series01_annexes == "out")
        series01_annexes[mask01] = str02_annex
    return series01_annexes

df01_hts['code'] = df01_hts['code'].astype(str)
df01_hts['annex_previo'] = map_annexes(df01_hts['code'], df04_old[df04_old['code_len'] <= 8])
df01_hts['annex_actual'] = map_annexes(df01_hts['code'], df05_new[df05_new['code_len'] <= 8])

df06_mapped_8 = df01_hts[(df01_hts['annex_previo'] != 'out') | (df01_hts['annex_actual'] != 'out')].copy()

df02_aux['code'] = df02_aux['code'].astype(str)
df02_aux['prefix_8'] = df02_aux['code'].str[:8]

df07_aux_matches = df02_aux[df02_aux['prefix_8'].isin(df06_mapped_8['code'])].copy()

list01_codes_expanded = df07_aux_matches['prefix_8'].unique()
df08_orphans = df06_mapped_8[~df06_mapped_8['code'].isin(list01_codes_expanded)].copy()

list02_exceptions = [
    "0305.59.00.01", "0305.69.50.01", "0305.69.60.01", "0714.40.10.01",
    "2903.59.10.10", "2903.99.80.01", "2905.49.50.01", "2918.17.00.01",
    "2922.12.00.01", "2926.90.48.01", "2933.99.17.01", "2933.99.97.01",
    "2934.99.90.01", "3808.91.25.01", "3808.91.50.01", "3808.99.95.01",
    "3901.90.55.01", "7013.37.10.01", "7013.99.40.01", "8477.90.45.01",
    "8505.90.75.01", "8510.90.30.01", "8529.90.24.01", "8531.90.90.01",
    "8540.11.24.01", "8540.11.28.01", "8540.11.44.01", "8540.11.48.01",
    "8542.33.00.01", "8543.70.93.01", "9005.90.80.01", "9006.91.00.01",
    "9007.91.80.01"
]
dict01_exc = {x.replace('.', '')[:8]: x.replace('.', '') for x in list02_exceptions}

df08_orphans['code_10'] = df08_orphans['code'].map(dict01_exc).fillna(df08_orphans['code'] + '00')

series02_manual_10d = df03_metals[df03_metals['code_len'] > 8]['code_clean']

df09_all_10 = pd.DataFrame({
    'code': pd.concat([df07_aux_matches['code'], df08_orphans['code_10'], series02_manual_10d])
}).drop_duplicates().reset_index(drop=True)

df09_all_10['annex_previo'] = map_annexes(df09_all_10['code'], df04_old)
df09_all_10['annex_actual'] = map_annexes(df09_all_10['code'], df05_new)

df10_final = df09_all_10[(df09_all_10['annex_previo'] != 'out') | (df09_all_10['annex_actual'] != 'out')].copy()

def get_status(row02):
    str03_prev = row02['annex_previo']
    str04_curr = row02['annex_actual']
    if str03_prev == 'out' and str04_curr != 'out':
        return 'Añadido'
    elif str03_prev != 'out' and str04_curr == 'out':
        return 'Quitado'
    elif str03_prev == str04_curr:
        return 'Mantenido'
    else:
        return 'Movido'

df10_final['status'] = df10_final.apply(get_status, axis=1)
df10_final = df10_final.sort_values(by='code').reset_index(drop=True)

In [9]:
df10_final.to_csv("../data/intermediate/metales_junio_10.csv", index=False)

In [7]:
import pandas as pd
from IPython.display import display, Markdown

df11_analisis = df10_final.copy()

display(Markdown("<br><h2>ANÁLISIS GLOBAL (CÓDIGOS ESTANDARIZADOS A 10 DÍGITOS)</h2><hr>"))

display(Markdown("#### 1. Resumen General de Estatus"))
df12_resumen = df11_analisis['status'].value_counts().reset_index()
df12_resumen.columns = ['Estatus', 'Total Códigos']
display(df12_resumen)

display(Markdown("#### 2. Desglose por Anexo (Añadidos, Mantenidos, Quitados)"))
df13_amq = df11_analisis[df11_analisis['status'].isin(['Añadido', 'Mantenido', 'Quitado'])].copy()
df13_amq['Anexo_Ref'] = df13_amq.apply(lambda r: r['annex_previo'] if r['status'] == 'Quitado' else r['annex_actual'], axis=1)

df14_resumen_anexos = pd.crosstab(
    index=df13_amq['Anexo_Ref'], 
    columns=df13_amq['status'], 
    margins=True, 
    margins_name='TOTAL GENERAL'
).fillna(0).astype(int)
df14_resumen_anexos.index.name = 'Anexo'
display(df14_resumen_anexos)

display(Markdown("#### 3. Códigos Movidos (Matriz de Transición)"))
df15_movidos = df11_analisis[df11_analisis['status'] == 'Movido']

if not df15_movidos.empty:
    df16_matriz_movidos = pd.crosstab(
        index=df15_movidos['annex_previo'], 
        columns=df15_movidos['annex_actual'], 
        margins=True, 
        margins_name="TOTAL"
    )
    df16_matriz_movidos.index.name = 'De (Anexo Previo)'
    df16_matriz_movidos.columns.name = 'A (Anexo Actual)'
    display(df16_matriz_movidos)
else:
    display(Markdown("-> *No hubo códigos movidos en este corte.*"))

display(Markdown("#### 4. Análisis de Composición por Anexo (Árbol de Detalle)"))
list03_anexos = sorted(list(set(df11_analisis['annex_previo'].unique()).union(set(df11_analisis['annex_actual'].unique())) - {'out'}))

for str01_anexo in list03_anexos:
    df17_añadidos = df11_analisis[(df11_analisis['annex_actual'] == str01_anexo) & (df11_analisis['status'] == 'Añadido')]
    df18_mantenidos = df11_analisis[(df11_analisis['annex_actual'] == str01_anexo) & (df11_analisis['status'] == 'Mantenido')]
    df19_quitados = df11_analisis[(df11_analisis['annex_previo'] == str01_anexo) & (df11_analisis['status'] == 'Quitado')]
    df20_entradas = df11_analisis[(df11_analisis['annex_actual'] == str01_anexo) & (df11_analisis['status'] == 'Movido')]
    df21_salidas = df11_analisis[(df11_analisis['annex_previo'] == str01_anexo) & (df11_analisis['status'] == 'Movido')]
    
    int01_total = len(df17_añadidos) + len(df18_mantenidos) + len(df20_entradas)
    
    print(f"📦 ANEXO {str01_anexo}")
    print(f"  ■ Composición Actual: {int01_total} códigos a 10 dígitos")
    print(f"    ├─ Mantenidos: {len(df18_mantenidos)}")
    print(f"    ├─ Añadidos:   {len(df17_añadidos)}")
    print(f"    └─ Entrantes:  {len(df20_entradas)}")
    
    if not df20_entradas.empty:
        for str02_origen in df20_entradas['annex_previo'].unique():
            int02_sub = len(df20_entradas[df20_entradas['annex_previo'] == str02_origen])
            print(f"       ↳ Desde {str02_origen}: {int02_sub}")
            
    print(f"  ■ Salidas (Impacto respecto a lista previa):")
    print(f"    ├─ Quitados:   {len(df19_quitados)}")
    print(f"    └─ Salientes:  {len(df21_salidas)}")
    
    if not df21_salidas.empty:
        for str03_destino in df21_salidas['annex_actual'].unique():
            int03_sub = len(df21_salidas[df21_salidas['annex_actual'] == str03_destino])
            print(f"       ↳ Hacia {str03_destino}: {int03_sub}")
    print("-" * 75)

<br><h2>ANÁLISIS GLOBAL (CÓDIGOS ESTANDARIZADOS A 10 DÍGITOS)</h2><hr>

#### 1. Resumen General de Estatus

,Estatus,Total Códigos
0,Mantenido,2028
1,Movido,176
2,Añadido,2
3,Quitado,1


#### 2. Desglose por Anexo (Añadidos, Mantenidos, Quitados)

status,Añadido,Mantenido,Quitado,TOTAL GENERAL
Anexo,,,,
Annex I-A,0,1205,0,1205
Annex I-B,0,514,1,515
Annex II,2,224,0,226
Annex III,0,85,0,85
TOTAL GENERAL,2,2028,1,2031


#### 3. Códigos Movidos (Matriz de Transición)

A (Anexo Actual),Annex I-B,Annex I-C,Annex III,TOTAL
De (Anexo Previo),,,,
Annex I-B,0,91,82,173
Annex II,3,0,0,3
TOTAL,3,91,82,176


#### 4. Análisis de Composición por Anexo (Árbol de Detalle)

📦 ANEXO Annex I-A
  ■ Composición Actual: 1205 códigos a 10 dígitos
    ├─ Mantenidos: 1205
    ├─ Añadidos:   0
    └─ Entrantes:  0
  ■ Salidas (Impacto respecto a lista previa):
    ├─ Quitados:   0
    └─ Salientes:  0
---------------------------------------------------------------------------
📦 ANEXO Annex I-B
  ■ Composición Actual: 517 códigos a 10 dígitos
    ├─ Mantenidos: 514
    ├─ Añadidos:   0
    └─ Entrantes:  3
       ↳ Desde Annex II: 3
  ■ Salidas (Impacto respecto a lista previa):
    ├─ Quitados:   1
    └─ Salientes:  173
       ↳ Hacia Annex III: 82
       ↳ Hacia Annex I-C: 91
---------------------------------------------------------------------------
📦 ANEXO Annex I-C
  ■ Composición Actual: 91 códigos a 10 dígitos
    ├─ Mantenidos: 0
    ├─ Añadidos:   0
    └─ Entrantes:  91
       ↳ Desde Annex I-B: 91
  ■ Salidas (Impacto respecto a lista previa):
    ├─ Quitados:   0
    └─ Salientes:  0
---------------------------------------------------------------------

In [5]:
import pandas as pd

clust01_final = pd.read_csv("../data/intermediate/metales_junio_10.csv", dtype={'code': str})
clust02_metals = pd.read_excel("../data/manual/metals_comparativa.xlsx", dtype={'code': str})

clust03_old = clust02_metals[clust02_metals['category'] == 'old'].copy()
clust03_old['clust04_clean'] = clust03_old['code'].str.replace('.', '', regex=False)

clust05_unique = clust03_old.drop_duplicates(subset=['clust04_clean']).copy()
clust05_unique['clust06_len'] = clust05_unique['clust04_clean'].str.len()
clust05_unique = clust05_unique.sort_values(by='clust06_len', ascending=False)

clust01_final['clust07_best'] = None
for _, clust08_row in clust05_unique.iterrows():
    clust09_prefix = clust08_row['clust04_clean']
    clust10_mask = clust01_final['code'].str.startswith(clust09_prefix) & clust01_final['clust07_best'].isna()
    clust01_final.loc[clust10_mask, 'clust07_best'] = clust09_prefix

def get_clust11_status(clust12_row):
    clust13_prev = clust12_row['annex_previo']
    clust14_curr = clust12_row['annex_actual']
    if clust13_prev == 'out' and clust14_curr != 'out': return 'Añadido'
    elif clust13_prev != 'out' and clust14_curr == 'out': return 'Quitado'
    elif clust13_prev == clust14_curr: return 'Mantenido'
    else: return 'Movido'

clust15_list = []

for _, clust16_row in clust03_old.iterrows():
    clust17_prefix = clust16_row['clust04_clean']
    clust18_anexo = clust16_row['annex']
    
    clust19_match = clust01_final[clust01_final['clust07_best'] == clust17_prefix].copy()
    
    if not clust19_match.empty:
        clust19_match['annex_previo'] = clust18_anexo
        clust19_match['status'] = clust19_match.apply(get_clust11_status, axis=1)
        clust15_list.append(clust19_match)

clust20_orphans = clust01_final[clust01_final['clust07_best'].isna()].copy()
if not clust20_orphans.empty:
    clust15_list.append(clust20_orphans)

clust21_final_ord = pd.concat(clust15_list, ignore_index=True)
clust21_final_ord = clust21_final_ord.drop(columns=['clust07_best'])

clust21_final_ord.to_csv("../data/intermediate/metales_junio_10_orden_old2.csv", index=False)